In [1]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

In [2]:
# Load dataset
dataset = load_dataset("dair-ai/emotion", "split")
train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()

In [3]:
# Define tokenizer
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")


In [4]:
# Tokenization function
train_texts = train_df["text"].tolist()
val_texts = val_df["text"].tolist()

# Tokenize texts properly
train_encodings = tokenizer(train_texts, padding="max_length", truncation=True, max_length=50)
val_encodings = tokenizer(val_texts, padding="max_length", truncation=True, max_length=50)

# Convert labels to tensors
train_labels = torch.tensor(train_df["label"].values)
val_labels = torch.tensor(val_df["label"].values)

In [5]:
# Define dataset class
class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

In [6]:
# Convert to PyTorch Dataset
train_dataset = EmotionDataset(train_encodings, train_labels)
val_dataset = EmotionDataset(val_encodings, val_labels)

In [7]:
# Load DistilBERT model for classification
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=6)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    save_steps=1000,
    save_total_limit=2,
    logging_dir="./logs"
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/anshulraj/BU/spring-25/GenAI/Projects/ai-music-generator/venv/lib/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [8]:
# Train model
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.240200,0.196991
2,0.146400,0.187922
3,0.099800,0.180255
4,0.067100,0.203786


TrainOutput(global_step=4000, training_loss=0.1785033187866211, metrics={'train_runtime': 600.1763, 'train_samples_per_second': 106.635, 'train_steps_per_second': 6.665, 'total_flos': 827980300800000.0, 'train_loss': 0.1785033187866211, 'epoch': 4.0})

In [9]:
# Save model
model.save_pretrained("models/distilbert_emotion")
tokenizer.save_pretrained("models/distilbert_emotion")

('models/distilbert_emotion/tokenizer_config.json',
 'models/distilbert_emotion/special_tokens_map.json',
 'models/distilbert_emotion/vocab.txt',
 'models/distilbert_emotion/added_tokens.json')